# ResNet-UNet

This notebook uses the shared preprocessing notebook before defining the model.

In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


from torchvision.models import resnet34, ResNet34_Weights

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")
print(f"Using torch version: {torch.__version__}")

Using device: cpu
Using torch version: 2.13.0


In [3]:
# running preprocessing notebook

%run ../shared/preprocessing.ipynb

In [ ]:
# Creating Data Loaders

train_loader, val_loader, test_loader, pos_weight = get_dataloaders( # type: ignore
    data_dir=DEFAULT_DATA_DIR, # type: ignore
    batch_size=32,
    augment_train=True,
    num_workers=0,
)

print("Input channels:", NUM_CHANNELS) # type: ignore
print("Positive-class weight:", pos_weight)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

I0000 00:00:1790316449.447161 3525042 tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144


Input channels: 12
Positive-class weight: 20.0
Train batches: 469
Validation batches: 59
Test batches: 53


In [5]:
inputs, targets, valid_mask = next(iter(train_loader))

print("Inputs: ", inputs.shape, inputs.dtype)
print("Targets: ", targets.shape, targets.dtype)
print("Valid mask: ", valid_mask.shape, valid_mask.dtype)

print("Target values: ", torch.unique(targets))
print("Mask values: ", torch.unique(valid_mask))

assert inputs.shape[1:] == (12, 64, 64)
assert targets.shape[1:] == (1, 64, 64)
assert valid_mask.shape == targets.shape
assert torch.isfinite(inputs).all()

print("Data contract verified successfully.")

Inputs:  torch.Size([32, 12, 64, 64]) torch.float32
Targets:  torch.Size([32, 1, 64, 64]) torch.float32
Valid mask:  torch.Size([32, 1, 64, 64]) torch.float32
Target values:  tensor([-1.,  0.,  1.])
Mask values:  tensor([0., 1.])
Data contract verified successfully.


# Build ResNet34–UNet

## Decoder Block

In [7]:
class DecoderBlock(nn.Module):
    def __init__(self, input_channels, skip_channels, output_channels):
        super().__init__()
        
        self.conv = nn.Sequential(
            nn.Conv2d(
                input_channels + skip_channels,
                output_channels,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                output_channels,
                output_channels,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x, skip):
        # Make x the sme height and width as skip feature
        
        x = F.interpolate(
            x,
            size=skip.shape[2:],
            mode="bilinear",
            align_corners=False,
        )
        
        # join detailed encoder features with upsampled decoder features
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

## Full Model

In [10]:
class ResNet34UNet(nn.Module):
    def __init__(self, input_channels=12):
        super().__init__()
        
        #Load renNet34 with Imagenet-trained weights
        self.encoder = resnet34(
            weights= ResNet34_Weights.IMAGENET1K_V1
        )
        
        # It's original first convolution accepts 3 RGB channels
        old_conv = self.encoder.conv1
        
        # Replace it with a convolution that accepts our 12 features
        new_conv = nn.Conv2d(
            input_channels,
            64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )
        
        # Initiate the new convolution from the pretrained RGB weights
        with torch.no_grad():
            rgb_average = old_conv.weight.mean(dim=1, keepdim=True)
            new_weights = rgb_average.repeat(
                1,
                input_channels,
                1,
                1
            ) * (3 / input_channels)
            
            new_conv.weight.copy_(new_weights)
            
            
        self.encoder.conv1 = new_conv
        
        # The classification layer is not needed for segmentation
        self.encoder.fc = nn.Identity() # type: ignore
        
        # Decorder: small features maps become layer step by step
        
        self.up4 = DecoderBlock(512, 256, 256)
        self.up3 = DecoderBlock(256, 128, 128)
        self.up2 = DecoderBlock(128, 64, 64)
        self.up1 = DecoderBlock(64, 64, 64)
        self.up0 = DecoderBlock(64, input_channels, 32)
        
        # One output value for each pixel: fire prediction logit.
        self.output_conv = nn.Conv2d(32, 1, kernel_size=1)
    
    
    def forward(self, x):
        original_inputs = x  # Save the 12-channel, 64 × 64 input

        skip0 = self.encoder.relu(
            self.encoder.bn1(self.encoder.conv1(x))
        )                                                  # 32 × 32

        skip1 = self.encoder.layer1(
            self.encoder.maxpool(skip0)
        )                                                  # 16 × 16
        skip2 = self.encoder.layer2(skip1)                 # 8 × 8
        skip3 = self.encoder.layer3(skip2)                 # 4 × 4
        bottom = self.encoder.layer4(skip3)                # 2 × 2

        x = self.up4(bottom, skip3)                        # 4 × 4
        x = self.up3(x, skip2)                             # 8 × 8
        x = self.up2(x, skip1)                             # 16 × 16
        x = self.up1(x, skip0)                             # 32 × 32
        x = self.up0(x, original_inputs)                   # 64 × 64

        return self.output_conv(x)
    

## Building Model

In [ ]:
model = ResNet34UNet(input_channels=NUM_CHANNELS).to(device) # type: ignore
model.eval()

with torch.no_grad():
    predictions = model(inputs[:2].to(device))

print("Model output:", predictions.shape)
print("Target shape:", targets[:2].shape)

Model output: torch.Size([2, 1, 64, 64])
Target shape: torch.Size([2, 1, 64, 64])


# Verify the loss

## Define the loss

In [12]:
def masked_bce_dice_loss(logits, targets, valid_mask, pos_weight):
    # BCE accepts only targets from 0 to 1.
    # Give uncertain (-1) pixels a temporary value of 0.
    safe_targets = torch.where(
        valid_mask == 1,
        targets,
        torch.zeros_like(targets),
    )

    # Give fire pixels the weight calculated from training data.
    fire_weight = torch.tensor(
        pos_weight,
        dtype=logits.dtype,
        device=logits.device,
    ).view(1, 1, 1, 1)

    # Calculate BCE separately for each pixel so we can mask pixels afterward.
    bce_per_pixel = F.binary_cross_entropy_with_logits(
        logits,
        safe_targets,
        pos_weight=fire_weight,
        reduction="none",
    )

    bce_loss = (
        (bce_per_pixel * valid_mask).sum()
        / valid_mask.sum().clamp_min(1)
    )

    # Dice compares predicted fire areas with actual fire areas.
    probabilities = torch.sigmoid(logits)
    overlap = (probabilities * safe_targets * valid_mask).sum()

    predicted_fire = (probabilities * valid_mask).sum()
    actual_fire = (safe_targets * valid_mask).sum()

    dice_score = (2 * overlap + 1) / (
        predicted_fire + actual_fire + 1
    )
    dice_loss = 1 - dice_score

    return bce_loss + 0.5 * dice_loss

## Prove that an uncertain pixel is ignored

In [ ]:
example_logits = torch.tensor(
    [[[[0.2, -0.3, 0.5]]]],
    requires_grad=True,
)
example_targets = torch.tensor([[[[1.0, 0.0, -1.0]]]])
example_mask = torch.tensor([[[[1.0, 1.0, 0.0]]]])

loss_before = masked_bce_dice_loss(
    example_logits, example_targets, example_mask, pos_weight
)

changed_logits = example_logits.detach().clone()
changed_logits[0, 0, 0, 2] = 10.0  # Change only the uncertain pixel

loss_after = masked_bce_dice_loss(
    changed_logits, example_targets, example_mask, pos_weight
)

assert torch.allclose(loss_before, loss_after)

loss_before.backward()
assert example_logits.grad[0, 0, 0, 2].item() == 0.0 # type: ignore

print("Uncertain-pixel masking works.")

Uncertain-pixel masking works.


## Check the loss with  real model and batch

In [15]:
model.eval()
model.zero_grad(set_to_none=True)

batch_inputs = inputs[:2].to(device)
batch_targets = targets[:2].to(device)
batch_mask = valid_mask[:2].to(device)

logits = model(batch_inputs)

loss = masked_bce_dice_loss(
    logits, batch_targets, batch_mask, pos_weight
)

assert torch.isfinite(loss)
loss.backward()

gradient = model.output_conv.weight.grad
assert gradient is not None
assert torch.isfinite(gradient).all()

print("Loss:", loss.item())

model.zero_grad(set_to_none=True)

Loss: 1.2330777645111084
